# speed from smooth data 

In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

#speed csv from smooth data

In [2]:

# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR  = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\xyz_Smooth"
OUTPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Speed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

fps = 2500
dt  = 1 / fps

begin_frame = None
end_frame   = None

# ==========================================================
# FIND FILES
# ==========================================================
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv")))
if not csv_files:
    raise FileNotFoundError("No *_SMOOTH.csv files found.")

# ==========================================================
# PROCESS
# ==========================================================
for path in csv_files:

    base = os.path.splitext(os.path.basename(path))[0]
    print("Processing:", base)

    # Extract Trial + RPM
    m = re.search(r"(Trial\d+_\d+rpm)", base)
    if not m:
        raise ValueError(f"Could not extract Trial+RPM from filename: {base}")
    trial_rpm = m.group(1)

    # ------------------------------------------------------
    # LOAD DATA
    # ------------------------------------------------------
    df = pd.read_csv(path)

    frames = df["frame"].values
    center = np.column_stack([
        df["center_X"].values,
        df["center_Y"].values,
        df["center_Z"].values
    ])

    # Optional slicing
    if begin_frame is not None and end_frame is not None:
        mask = (frames >= begin_frame) & (frames <= end_frame)
        frames = frames[mask]
        center = center[mask]

    # ------------------------------------------------------
    # 1) DISPLACEMENT (explicit)
    # ------------------------------------------------------
    displacement = np.full_like(center, np.nan)

    displacement[1:] = center[1:] - center[:-1]
    displacement[0]  = np.nan

    # Components
    disp_xy = displacement[:, [0, 1]]
    disp_yz = displacement[:, [1, 2]]

    disp_mag      = np.linalg.norm(displacement, axis=1)
    disp_mag_xy   = np.linalg.norm(disp_xy, axis=1)
    disp_mag_yz   = np.linalg.norm(disp_yz, axis=1)

    # ------------------------------------------------------
    # 2) VELOCITY (from displacement)
    # ------------------------------------------------------
    velocity = displacement / dt

    vel_xy = velocity[:, [0, 1]]
    vel_yz = velocity[:, [1, 2]]

    vel_mag    = np.linalg.norm(velocity, axis=1)
    vel_mag_xy = np.linalg.norm(vel_xy, axis=1)
    vel_mag_yz = np.linalg.norm(vel_yz, axis=1)

    # ------------------------------------------------------
    # 3) SPEED (scalar velocity magnitude)
    # ------------------------------------------------------
    speed      = vel_mag
    speed_xy   = vel_mag_xy
    speed_yz   = vel_mag_yz

    # ------------------------------------------------------
    # SAVE CSV
    # ------------------------------------------------------
    out_df = pd.DataFrame({
        "frame": frames,

        # Displacement
        "displacement_m": disp_mag,
        "displacement_xy_m": disp_mag_xy,
        "displacement_yz_m": disp_mag_yz,

        # Velocity
        "velocity_m_per_s": vel_mag,
        "velocity_xy_m_per_s": vel_mag_xy,
        "velocity_yz_m_per_s": vel_mag_yz,

        # Speed (explicit)
        "speed_m_per_s": speed,
        "speed_xy_m_per_s": speed_xy,
        "speed_yz_m_per_s": speed_yz
    })

    out_path = os.path.join(OUTPUT_DIR, f"{trial_rpm}_SPEED.csv")
    out_df.to_csv(out_path, index=False)

    print("Saved:", out_path)

print("\n speed CSVs generated explicitly.")


Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_SMOOTH
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Speed\Trial2_180rpm_SPEED.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_SMOOTH
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Speed\Trial2_200rpm_SPEED.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_SMOOTH
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Speed\Trial3_180rpm_SPEED.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_SMOOTH
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Speed\Trial4_400rpm_SPEED.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_SMOOTH
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\

#save and display speed plots


In [3]:


# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\Speed"
BASE_TRIAL_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data"

FPS = 2500

# ==========================================================
# FIND SPEED FILES
# ==========================================================
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_SPEED.csv")))
if not csv_files:
    raise FileNotFoundError("No *_SPEED.csv files found.")

# ==========================================================
# PLOTTING
# ==========================================================
for path in csv_files:
    base = os.path.splitext(os.path.basename(path))[0]
    print("Plotting:", base)

    # ------------------------------------------------------
    # Extract Trial + RPM
    # ------------------------------------------------------
    m = re.search(r"(Trial\d+_\d+rpm)", base)
    if not m:
        raise ValueError(f"Could not extract Trial+RPM from filename: {base}")
    trial_name = m.group(1)

    # ------------------------------------------------------
    # Create trial-specific SPEED_PLOTS folder
    # ------------------------------------------------------
    save_dir = os.path.join(BASE_TRIAL_DIR, trial_name, "SPEED_PLOTS")
    os.makedirs(save_dir, exist_ok=True)

    # ------------------------------------------------------
    # Load data
    # ------------------------------------------------------
    df = pd.read_csv(path)

    frames = df["frame"].values
    time_sec = frames / FPS

    speed    = df["speed_m_per_s"].values
    speed_xy = df["speed_xy_m_per_s"].values
    speed_yz = df["speed_yz_m_per_s"].values

    # ======================================================
    #  RESULTANT SPEED vs TIME
    # ======================================================
    plt.figure(figsize=(14, 4))
    plt.plot(time_sec, speed, linewidth=2, color="black")

    plt.xlabel("Time (s)", fontsize=24)
    plt.ylabel("Speed (m/s)", fontsize=24)
    plt.tick_params(axis="both", labelsize=18)
    plt.title("Resultant Speed vs Time", fontsize=24)
    plt.grid(True)
    plt.tight_layout()

    out1 = os.path.join(save_dir, f"{base}_resultant_speed.png")
    plt.savefig(out1, dpi=300)
    plt.close()

    # ======================================================
    #  SPEED COMPONENTS vs TIME
    # ======================================================
    plt.figure(figsize=(14, 4))
    plt.plot(time_sec, speed, label="Resultant (3D)", color="black", linewidth=2)
    plt.plot(time_sec, speed_xy, label="Horizontal (XY)", color="purple", alpha=0.8)
    plt.plot(time_sec, speed_yz, label="Vertical (YZ)", color="teal", alpha=0.8)

    plt.xlabel("Time (s)", fontsize=24)
    plt.ylabel("Speed (m/s)", fontsize=24)
    plt.tick_params(axis="both", labelsize=18)
    plt.title("Speed Components vs Time", fontsize=24)
    plt.legend(fontsize=16)
    plt.grid(True)
    plt.tight_layout()

    out2 = os.path.join(save_dir, f"{base}_speed_components.png")
    plt.savefig(out2, dpi=300)
    plt.close()

    print("Saved:")
    print(" ", out1)
    print(" ", out2)

print("\n Speed plots saved trial + RPM wise.")


Plotting: Trial2_180rpm_SPEED
Saved:
  D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_180rpm\SPEED_PLOTS\Trial2_180rpm_SPEED_resultant_speed.png
  D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_180rpm\SPEED_PLOTS\Trial2_180rpm_SPEED_speed_components.png
Plotting: Trial2_200rpm_SPEED
Saved:
  D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_200rpm\SPEED_PLOTS\Trial2_200rpm_SPEED_resultant_speed.png
  D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial2_200rpm\SPEED_PLOTS\Trial2_200rpm_SPEED_speed_components.png
Plotting: Trial3_180rpm_SPEED
Saved:
  D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial3_180rpm\SPEED_PLOTS\Trial3_180rpm_SPEED_resultant_speed.png
  D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial3_180rpm\SPEED_PLOTS\Trial3_180rpm_SPEED_speed_components.png
Plotting: Trial4_400rpm_SPEED
Saved:
  D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data\Trial4_400rpm\SPEED_PLOTS\Trial4_400rpm_SPEED_resultant_speed.png
  D:\Nishka_Analysis\Analysis_